# City Embeddings Creation

This notebook computes per-city embeddings from the base case datasets under `data.symlink.1761915057/inductive_data/links_and_stats/basecases_mean`.

For each city, it calculates:
- Population (distinct `person_id` in `eqasim_trips.csv`)
- Network density (from `basecase_average_output_links.geojson` using `from_node` and `to_node`)
- Total number of trips (row count in `eqasim_trips.csv`)
- Average total travel times by mode (from `basecase_average_trips.csv`): car, car_passenger, bicycle, outside, pt, walk

Outputs a CSV at `data.symlink.1761915057/city_embedding.csv` with rows as cities and columns as characteristics.


In [1]:
from __future__ import annotations
import os
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd

# Resolve dataset base directory robustly from current working directory
RELATIVE_DATA_PATH = Path("data.symlink.1761915057/inductive_data/links_and_stats/basecases_mean")

def resolve_base_dir() -> Path:
    candidates = [
        RELATIVE_DATA_PATH,
        Path.cwd() / RELATIVE_DATA_PATH,
        Path.cwd().parent / RELATIVE_DATA_PATH,
        Path.cwd().parent.parent / RELATIVE_DATA_PATH,
    ]
    for cand in candidates:
        if cand.exists():
            return cand
    tried = "\n".join(str(c.resolve()) for c in candidates)
    raise AssertionError(f"Base directory not found. Tried paths:\n{tried}")

BASE_DIR = resolve_base_dir()
# OUTPUT_CSV should live alongside the data.symlink root
OUTPUT_CSV = BASE_DIR.parents[2] / "city_embedding.csv"

print(f"Using BASE_DIR: {BASE_DIR}")
print(f"Will write OUTPUT_CSV to: {OUTPUT_CSV}")

# Density configuration: 'directed' or 'undirected'
NETWORK_DENSITY_MODE = 'directed'

Using BASE_DIR: /home/enatterer/Development/elena_gnn_predicting_effects_of_traffic_policies/data.symlink.1761915057/inductive_data/links_and_stats/basecases_mean
Will write OUTPUT_CSV to: /home/enatterer/Development/elena_gnn_predicting_effects_of_traffic_policies/data.symlink.1761915057/city_embedding.csv


In [2]:
def list_cities(base_dir: Path) -> List[str]:
    cities: List[str] = []
    for entry in sorted(base_dir.iterdir()):
        if entry.is_dir():
            # Require the three expected files to exist
            geojson = entry / "basecase_average_output_links.geojson"
            avg_trips = entry / "basecase_average_trips.csv"
            eqasim = entry / "eqasim_trips.csv"
            if geojson.exists() and avg_trips.exists() and eqasim.exists():
                cities.append(entry.name)
    return cities

cities = list_cities(BASE_DIR)
print(f"Found {len(cities)} cities")
print(cities[:10])


Found 16 cities
['aschaffenburg', 'augsburg', 'bamberg', 'bayreuth', 'erlangen', 'fuerth', 'ingolstadt', 'kempten', 'landshut', 'muenchen']


In [3]:
def load_eqasim_trips(city_dir: Path) -> pd.DataFrame:
    eqasim_path = city_dir / "eqasim_trips.csv"
    # File is semicolon-separated
    df = pd.read_csv(eqasim_path, sep=";", low_memory=False)
    if "person_id" not in df.columns:
        # try common variants
        candidates = [c for c in df.columns if c.lower().replace(" ", "_") in {"person_id", "personid", "pid"}]
        if candidates:
            df = df.rename(columns={candidates[0]: "person_id"})
        else:
            raise ValueError(f"Could not find person_id column in {eqasim_path}")
    return df[["person_id"]]


def load_average_trips(city_dir: Path) -> pd.DataFrame:
    avg_path = city_dir / "basecase_average_trips.csv"
    df = pd.read_csv(avg_path)
    # Normalize column names
    df.columns = [c.strip() for c in df.columns]
    return df


def load_links_from_to(city_dir: Path) -> Tuple[List[str], List[str]]:
    geojson_path = city_dir / "basecase_average_output_links.geojson"
    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    features = data.get("features", [])
    from_nodes: List[str] = []
    to_nodes: List[str] = []
    for feat in features:
        props = feat.get("properties", {})
        fnode = props.get("from_node")
        tnode = props.get("to_node")
        if fnode is not None and tnode is not None:
            from_nodes.append(str(fnode))
            to_nodes.append(str(tnode))
    return from_nodes, to_nodes


In [4]:
def compute_population_and_trips(eqasim_df: pd.DataFrame) -> Tuple[int, int]:
    s = (
        eqasim_df["person_id"]
        .astype(str)
        .str.strip()
    )
    s = s[s.notna() & (s != "")]
    population = int(s.nunique())
    total_trips = int(len(eqasim_df))
    return population, total_trips


def compute_network_density(from_nodes: List[str], to_nodes: List[str]) -> float:
    if not from_nodes and not to_nodes:
        return 0.0
    unique_nodes = set(from_nodes) | set(to_nodes)
    n = len(unique_nodes)
    if n <= 1:
        return 0.0
    # Treat as directed graph without self-loops, deduplicate edges
    edges = set(zip(from_nodes, to_nodes))
    # Exclude self-loops if any
    edges = {(u, v) for (u, v) in edges if u != v}
    m = len(edges)
    density = m / (n * (n - 1))
    return float(density)


def _normalize(s: str) -> str:
    return s.strip().lower().replace(" ", "_")


def extract_travel_time_averages(avg_trips_df: pd.DataFrame) -> Dict[str, Optional[float]]:
    # File has one row per mode with columns: mode, avg_total_travel_time, ...
    # Build a mapping from mode -> avg_total_travel_time
    mode_col = None
    time_col = None
    cols_norm = {_normalize(c): c for c in avg_trips_df.columns}
    for cand in ["mode"]:
        if cand in cols_norm:
            mode_col = cols_norm[cand]
    for cand in ["avg_total_travel_time", "average_total_travel_time"]:
        if cand in cols_norm:
            time_col = cols_norm[cand]
    if mode_col is None or time_col is None:
        return {k: None for k in ["car","car_passenger","bicycle","outside","pt","walk"]}

    df = avg_trips_df[[mode_col, time_col]].copy()
    df[mode_col] = df[mode_col].astype(str).str.strip().str.lower()
    wanted = ["car", "car_passenger", "bicycle", "outside", "pt", "walk"]
    out: Dict[str, Optional[float]] = {}
    for m in wanted:
        row = df.loc[df[mode_col] == m]
        if not row.empty:
            try:
                out[m] = float(row.iloc[0][time_col])
            except Exception:
                out[m] = None
        else:
            out[m] = None
    return out


In [5]:
records: List[Dict[str, Optional[float]]] = []

for city in cities:
    city_dir = BASE_DIR / city
    # Load data
    eqasim_df = load_eqasim_trips(city_dir)
    avg_trips_df = load_average_trips(city_dir)
    from_nodes, to_nodes = load_links_from_to(city_dir)

    # Compute metrics
    population, total_trips = compute_population_and_trips(eqasim_df)
    density = compute_network_density(from_nodes, to_nodes)
    averages = extract_travel_time_averages(avg_trips_df)

    record: Dict[str, Optional[float]] = {
        "city": city,
        "Population": int(population),
        "Network density": float(density),
        "Total number of trips": int(total_trips),
        "average total travel time car": averages.get("car"),
        "average total travel time car_passenger": averages.get("car_passenger"),
        "average total travel time bicycle": averages.get("bicycle"),
        "average total travel time outside": averages.get("outside"),
        "average total travel time pt": averages.get("pt"),
        "average total travel time walk": averages.get("walk"),
    }
    records.append(record)

print(f"Computed metrics for {len(records)} cities")


Computed metrics for 16 cities


In [6]:
# Assemble DataFrame with required column order and save
city_df = pd.DataFrame.from_records(records)

# 1) Rename Population -> Person_eqasim
if "Population" in city_df.columns:
    city_df = city_df.rename(columns={"Population": "Person_eqasim"})

# 2) Add real-world Population (2025) from the Bavarian statistics report (cities only, no Landkreis)
# Source: https://www.statistik.bayern.de/mam/produkte/veroffentlichungen/statistische_berichte/a1200c_202541.pdf
# Option A (preferred): provide data.symlink.1761915057/population_2025.csv with columns: city,population
# Option B (fallback): use built-in defaults below.

# Fallback defaults (override by CSV if present). Keys match folder names (lowercase, ascii).
population_2025_by_city = {
    # Exact city populations (kreisfreie Städte) per 2025 report
    "aschaffenburg": 72219,
    "augsburg": 307120,
    "bamberg": 77962,
    "bayreuth": 75590,
    "erlangen": 119781,
    "fuerth": 131595,
    "ingolstadt": 142214,
    "kempten": 72290,      # Kempten (Allgaeu)
    "landshut": 77965,
    "muenchen": 1617191,   # München
    "neuulm": 63581,       # Neu-Ulm
    "nuernberg": 554835,   # Nürnberg
    "regensburg": 156778,
    "rosenheim": 63853,
    "schweinfurt": 53523,
    "wuerzburg": 127988,
}

# Try override from CSV if available
pop_csv = BASE_DIR.parents[2] / "population_2025.csv"
if pop_csv.exists():
    df_pop = pd.read_csv(pop_csv)
    # Expect columns: city (matching our folder names), population (int)
    if {"city", "population"}.issubset(set(df_pop.columns)):
        override = {
            str(r["city"]).strip().lower(): int(r["population"]) for _, r in df_pop.iterrows()
            if pd.notna(r["city"]) and pd.notna(r["population"])
        }
        population_2025_by_city.update(override)
    else:
        print(f"Warning: {pop_csv} missing required columns 'city' and 'population'; using defaults.")

missing = []
real_pop = []
for city in city_df["city"].tolist():
    val = population_2025_by_city.get(city)
    if val is None:
        missing.append(city)
    real_pop.append(val)
city_df["Population"] = real_pop
if missing:
    print("Warning: Missing Population(2025) for:", sorted(set(missing)))

# 3) Add additional columns from the provided table
# Table data with city names matching folder names (lowercase)
# Note: decimal separator in table is comma, convert to period
table_data = {
    "muenchen": {
        "population_kreisfreistadt": 1512491,
        "kreisfreistadt_area": 310.70,
        "population_density": 4868.01,
        "driving_license_ownership": 832387,  # 832,387 (comma is thousands separator for counts)
        "network_density_table": 0.000086,
        "average_betweenness": 0.004634,
        "mean_edge_centrality": 0.000105,
    },
    "augsburg": {
        "population_kreisfreistadt": 301033,
        "kreisfreistadt_area": 146.85,
        "population_density": 2049.94,
        "driving_license_ownership": 164260,
        "network_density_table": 0.000250,
        "average_betweenness": 0.009449,
        "mean_edge_centrality": 0.000195,
    },
    "nuernberg": {
        "population_kreisfreistadt": 523026,
        "kreisfreistadt_area": 186.44,
        "population_density": 2805.33,
        "driving_license_ownership": 283033,  # 283,033 (comma is thousands separator for counts)
        "network_density_table": 0.000158,
        "average_betweenness": 0.007379,
        "mean_edge_centrality": 0.000132,
    },
    "ingolstadt": {
        "population_kreisfreistadt": 141029,
        "kreisfreistadt_area": 133.35,
        "population_density": 1057.59,
        "driving_license_ownership": 84154,  # 84,154 (comma is thousands separator for counts)
        "network_density_table": 0.000435,
        "average_betweenness": 0.015684,
        "mean_edge_centrality": 0.000153,
    },
    "regensburg": {
        "population_kreisfreistadt": 157443,
        "kreisfreistadt_area": 80.86,
        "population_density": 1947.11,
        "driving_license_ownership": 208192,  # 208,192 (comma is thousands separator for counts)
        "network_density_table": 0.000391,
        "average_betweenness": 0.013233,
        "mean_edge_centrality": 0.000185,
    },
    "wuerzburg": {
        "population_kreisfreistadt": 127810,
        "kreisfreistadt_area": 87.60,
        "population_density": 1459.02,
        "driving_license_ownership": 149762,  # 149,762 (comma is thousands separator for counts)
        "network_density_table": 0.000377,
        "average_betweenness": 0.011508,
        "mean_edge_centrality": 0.000203,
    },
    "aschaffenburg": {
        "population_kreisfreistadt": 72444,
        "kreisfreistadt_area": 62.45,
        "population_density": 1160.03,
        "driving_license_ownership": 42137,  # 42,137 (comma is thousands separator for counts)
        "network_density_table": 0.000734,
        "average_betweenness": 0.016015,
        "mean_edge_centrality": 0.000268,
    },
    "bamberg": {
        "population_kreisfreistadt": 79935,
        "kreisfreistadt_area": 54.62,
        "population_density": 1463.47,
        "driving_license_ownership": 142759,  # 142,759 (comma is thousands separator for counts)
        "network_density_table": 0.000299,
        "average_betweenness": 0.015718,
        "mean_edge_centrality": 0.000317,
    },
    "bayreuth": {
        "population_kreisfreistadt": 74506,
        "kreisfreistadt_area": 66.89,
        "population_density": 1113.86,
        "driving_license_ownership": 110440,
        "network_density_table": 0.000640,
        "average_betweenness": 0.018992,
        "mean_edge_centrality": 0.000238,
    },
    "erlangen": {
        "population_kreisfreistadt": 116562,
        "kreisfreistadt_area": 76.96,
        "population_density": 1514.58,
        "driving_license_ownership": 155374,  # 155,374 (comma is thousands separator for counts)
        "network_density_table": 0.000449,
        "average_betweenness": 0.015376,
        "mean_edge_centrality": 0.000196,
    },
    "fuerth": {
        "population_kreisfreistadt": 131433,
        "kreisfreistadt_area": 63.35,
        "population_density": 2074.71,
        "driving_license_ownership": 68859,  # 68,859 (comma is thousands separator for counts)
        "network_density_table": 0.000650,
        "average_betweenness": 0.018287,
        "mean_edge_centrality": 0.000191,
    },
    "kempten": {
        "population_kreisfreistadt": 70056,
        "kreisfreistadt_area": 63.28,
        "population_density": 1107.08,
        "driving_license_ownership": 70074,  # 70,074 (comma is thousands separator for counts)
        "network_density_table": 0.000729,
        "average_betweenness": 0.018252,
        "mean_edge_centrality": 0.000259,
    },
    "landshut": {
        "population_kreisfreistadt": 75457,
        "kreisfreistadt_area": 65.83,
        "population_density": 1146.24,
        "driving_license_ownership": 153662,  # 153,662 (comma is thousands separator for counts)
        "network_density_table": 0.000782,
        "average_betweenness": 0.022464,
        "mean_edge_centrality": 0.000218,
    },
    "rosenheim": {
        "population_kreisfreistadt": 64403,
        "kreisfreistadt_area": 37.22,
        "population_density": 1730.33,
        "driving_license_ownership": 206671,  # 206,671 (comma is thousands separator for counts)
        "network_density_table": 0.000977,
        "average_betweenness": 0.023865,
        "mean_edge_centrality": 0.000302,
    },
    "schweinfurt": {
        "population_kreisfreistadt": 54675,
        "kreisfreistadt_area": 35.70,
        "population_density": 1531.51,
        "driving_license_ownership": 113912,  # 113,912 (comma is thousands separator for counts)
        "network_density_table": 0.000854,
        "average_betweenness": 0.019796,
        "mean_edge_centrality": 0.000314,
    },
    "neuulm": {
        "population_kreisfreistadt": 180425,
        "kreisfreistadt_area": 515.84,
        "population_density": 349.77,
        "driving_license_ownership": 111926,  # 111,926 (comma is thousands separator for counts)
        "network_density_table": 0.000200,
        "average_betweenness": 0.010726,
        "mean_edge_centrality": 0.000069,
    },
}

# Add columns from table data
# Update Population with population(kreisfreistadt) from table
pop_from_table = city_df["city"].map(lambda x: table_data.get(x, {}).get("population_kreisfreistadt"))
city_df["Population"] = pop_from_table.fillna(city_df["Population"])

# Add area column (kreisfreistadt_area -> area)
city_df["area"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("kreisfreistadt_area"))

# Add population_density column
city_df["population_density"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("population_density"))

# Replace Network density with network_density from table
city_df["network_density"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("network_density_table"))
# Remove the old "Network density" column (we use network_density from table instead)
if "Network density" in city_df.columns:
    city_df = city_df.drop(columns=["Network density"])

# Add average_betweenness column
city_df["average_betweenness"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("average_betweenness"))

# Add mean_edge_centrality column
city_df["mean_edge_centrality"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("mean_edge_centrality"))

# Add driving_licence_ownership column
city_df["driving_licence_ownership"] = city_df["city"].map(lambda x: table_data.get(x, {}).get("driving_license_ownership"))

# Calculate percentage_driving_license = driving_licence_ownership / Population
city_df["percentage_driving_license"] = (
    city_df["driving_licence_ownership"] /
    city_df["Population"].replace({0: pd.NA})
)

# Column order with new columns
columns = [
    "city",
    "Person_eqasim",
    "Population",
    "area",
    "population_density",
    "network_density",
    "average_betweenness",
    "mean_edge_centrality",
    "driving_licence_ownership",
    "percentage_driving_license",
    "Total number of trips",
    "average total travel time car",
    "average total travel time car_passenger",
    "average total travel time bicycle",
    "average total travel time outside",
    "average total travel time pt",
    "average total travel time walk",
]

# Ensure all columns exist in order
for c in columns:
    if c not in city_df.columns:
        city_df[c] = None
city_df = city_df[columns]

# Sort by city name for reproducibility
city_df = city_df.sort_values("city").reset_index(drop=True)

# Write to CSV
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
city_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved embeddings to {OUTPUT_CSV}")
city_df.head()


Saved embeddings to /home/enatterer/Development/elena_gnn_predicting_effects_of_traffic_policies/data.symlink.1761915057/city_embedding.csv


,city,Person_eqasim,Population,area,population_density,network_density,average_betweenness,mean_edge_centrality,driving_licence_ownership,percentage_driving_license,Total number of trips,average total travel time car,average total travel time car_passenger,average total travel time bicycle,average total travel time outside,average total travel time pt,average total travel time walk
0,aschaffenburg,24735,72444,62.45,1160.03,0.000734,0.016015,0.000268,42137,0.581649,92083,765.080474,782.307741,957.164877,0.970589,3490.442528,996.885583
1,augsburg,61369,301033,146.85,2049.94,0.000250,0.009449,0.000195,164260,0.545654,234232,1352.819360,752.984886,980.698990,1.782931,3522.988980,1400.061863
2,bamberg,28812,79935,54.62,1463.47,0.000299,0.015718,0.000317,142759,1.785939,107451,603.370714,593.897835,1060.031799,1.772884,5098.448457,1625.094358
3,bayreuth,21290,74506,66.89,1113.86,0.000640,0.018992,0.000238,110440,1.482297,80246,819.619895,680.183352,1013.335122,2.939046,4881.925150,1626.797604
4,erlangen,40934,116562,76.96,1514.58,0.000449,0.015376,0.000196,155374,1.332973,182557,773.283153,511.952929,904.682102,0.954388,3176.854024,1620.185007


In [7]:
# Quick validation: check one city end-to-end
if len(cities) > 0:
    test_city = cities[0]
    print(f"Validating city: {test_city}")
    city_dir = BASE_DIR / test_city
    eqasim_df = load_eqasim_trips(city_dir)
    avg_trips_df = load_average_trips(city_dir)
    from_nodes, to_nodes = load_links_from_to(city_dir)

    population, total_trips = compute_population_and_trips(eqasim_df)
    density = compute_network_density(from_nodes, to_nodes)
    averages = extract_travel_time_averages(avg_trips_df)

    print({
        "population": population,
        "total_trips": total_trips,
        "density": density,
        **{f"avg_{k}": v for k, v in averages.items()},
    })

city_df.head(10)

Validating city: aschaffenburg
{'population': 24735, 'total_trips': 92083, 'density': 0.00014267661428323523, 'avg_car': 765.0804736726765, 'avg_car_passenger': 782.3077410640925, 'avg_bicycle': 957.1648765615606, 'avg_outside': 0.9705890798415752, 'avg_pt': 3490.4425284053464, 'avg_walk': 996.8855832081224}


,city,Person_eqasim,Population,area,population_density,network_density,average_betweenness,mean_edge_centrality,driving_licence_ownership,percentage_driving_license,Total number of trips,average total travel time car,average total travel time car_passenger,average total travel time bicycle,average total travel time outside,average total travel time pt,average total travel time walk
0,aschaffenburg,24735,72444,62.45,1160.03,0.000734,0.016015,0.000268,42137,0.581649,92083,765.080474,782.307741,957.164877,0.970589,3490.442528,996.885583
1,augsburg,61369,301033,146.85,2049.94,0.000250,0.009449,0.000195,164260,0.545654,234232,1352.819360,752.984886,980.698990,1.782931,3522.988980,1400.061863
2,bamberg,28812,79935,54.62,1463.47,0.000299,0.015718,0.000317,142759,1.785939,107451,603.370714,593.897835,1060.031799,1.772884,5098.448457,1625.094358
3,bayreuth,21290,74506,66.89,1113.86,0.000640,0.018992,0.000238,110440,1.482297,80246,819.619895,680.183352,1013.335122,2.939046,4881.925150,1626.797604
4,erlangen,40934,116562,76.96,1514.58,0.000449,0.015376,0.000196,155374,1.332973,182557,773.283153,511.952929,904.682102,0.954388,3176.854024,1620.185007
5,fuerth,33705,131433,63.35,2074.71,0.000650,0.018287,0.000191,68859,0.523910,134389,500.837579,476.178789,778.375883,0.883941,2843.680055,1165.045517
6,ingolstadt,22817,141029,133.35,1057.59,0.000435,0.015684,0.000153,84154,0.596714,95513,1266.759412,450.828178,789.036505,1.254074,2484.344597,1294.291703
7,kempten,12232,70056,63.28,1107.08,0.000729,0.018252,0.000259,70074,1.000257,45664,439.445567,352.281995,723.668036,0.402833,1942.673084,1545.957440
8,landshut,27405,75457,65.83,1146.24,0.000782,0.022464,0.000218,153662,2.036418,99642,755.912264,756.740845,1113.908392,1.482025,6886.927595,2105.102838
9,muenchen,216679,1512491,310.70,4868.01,0.000086,0.004634,0.000105,832387,0.550342,863717,1761.046645,933.393435,806.811329,1.196653,2528.375236,1144.368626


In [8]:
# Diagnostics: trips per person by city (sanity check)
sanity = city_df.copy()
sanity["trips_per_person"] = sanity["Total number of trips"] / sanity["Population"].replace({0: pd.NA})
print(sanity[["city", "Population", "Total number of trips", "trips_per_person"]])


             city  Population  Total number of trips  trips_per_person
0   aschaffenburg       72444                  92083          1.271092
1        augsburg      301033                 234232          0.778094
2         bamberg       79935                 107451          1.344230
3        bayreuth       74506                  80246          1.077041
4        erlangen      116562                 182557          1.566179
5          fuerth      131433                 134389          1.022491
6      ingolstadt      141029                  95513          0.677258
7         kempten       70056                  45664          0.651821
8        landshut       75457                  99642          1.320514
9        muenchen     1512491                 863717          0.571056
10         neuulm      180425                  63970          0.354552
11      nuernberg      523026                 434981          0.831662
12     regensburg      157443                 157044          0.997466
13    